### STEP file check - DAGMC generation

With this script I check .stp files for DAGMC (.h5m and .vtk) generation.
The script is divided in two sections:
1) DAGMC generation in one  
- First I convert them to DAGMC .h5m using CadToDagmc;
- Second I use DAGMC tools `overlap_check` and `check_watertight` to detect overlaps and leaky volumes;
- Third I convert to .vtk for later unstructured mesh
2) DAGMC generation per part
- First I convert to .h5m part by part
- I check for watertight and overlaps each parts
- I merge the single part.h5m files into one merged.h5m file
- I check the merged.h5m for watertight and overlaps
- Second I convert to .vtk part by part
- I merge the part.vtk files into on merged.vtk file

In [ ]:
# First import cad_to_dagmc and configure the files paths

from cad_to_dagmc import CadToDagmc
from pathlib import Path
import subprocess
import shutil

# ----------------------------
# Paths
# ----------------------------

base_folder = Path("/home/riccardo/neutronics/Geometry/Geometry_files/W7X_M1_GeomCheck")

step_folder = base_folder / "step_files"

h5m_folder = base_folder / "h5m_files"
vtk_folder = base_folder / "vtk_files"

dagmc_tools_path = Path("/home/riccardo/dagmc-tools/bin")

# ----------------------------
# Collect STEP files
# ----------------------------

step_files = sorted(step_folder.glob("*.stp"))

print(f"Found {len(step_files)} STEP files")

# Show indexed list
for i, file in enumerate(step_files):
    print(f"[{i}] {file.name}")

Choose file

In [ ]:
index = 0  # change manually

stp_file = step_files[index]
h5m_file = h5m_folder / stp_file.with_suffix(".h5m").name
vtk_file = vtk_folder / stp_file.with_suffix(".vtk").name

print(f"Processing: {stp_file.name}")

Material tag definition

In [ ]:
material_tag = f"{stp_file.stem}_material"

print(f"Assigned material tag: {material_tag}")
change = input("Do you want to change it? [y/N]: ").strip().lower()

if change == "y":
    material_tag = input("Enter new material tag: ").strip()

print(f"Using material tag: {material_tag}")

Make an instance of CadToDagmc and add the step file to the model:

In [ ]:
my_CadToDagmc = CadToDagmc()

my_CadToDagmc.add_stp_file(
    filename=str(stp_file),
    scale_factor=0.1  # mm → cm
)

## 1) DAGMC generation in one

Assign material tags

In [ ]:
print("Number of cad_to_dagmc parts:", len(my_CadToDagmc.parts))

num_volumes = len(my_CadToDagmc.parts)

my_CadToDagmc.material_tags = [material_tag] * num_volumes

print("Material tag assigned:", material_tag)

Now export the DAGMC .h5m geometry file 

In [ ]:
my_CadToDagmc.export_dagmc_h5m_file(
    filename=str(h5m_file),
    min_mesh_size=0.5,
    max_mesh_size=2.0,
)

print("DAGMC file created:", h5m_file.name)

Check .h5m file for watertight using DAGMC tool check_watertight

In [ ]:
cmd = [
    str(dagmc_tools_path / "check_watertight"),
    str(h5m_file)
]

subprocess.run(cmd)

Check .h5m file for overlap using DAGMC tool overlap_check

In [ ]:
cmd = [
    str(dagmc_tools_path / "overlap_check"),
    str(h5m_file)
]

subprocess.run(cmd)

Now export the DAGMC .vtk geometry file

In [ ]:
my_CadToDagmc.export_unstructured_mesh_file(
    filename=str(vtk_file),
    min_mesh_size=5.0,
    max_mesh_size=20.0,
)

print("VTK file created:", vtk_file.name)

## 2) DAGMC generation per part

Export step as h5m part by part

In [ ]:
# Create per-part DAGMC .h5m files and run checks

parts_h5m_files = []
failed_parts_h5m = []

for i, part in enumerate(my_CadToDagmc.parts):
    try:
        temp = CadToDagmc()
        temp.parts = [part]
        temp.material_tags = [material_tag] 
        part_h5m = h5m_folder / f"{stp_file.stem}_part_{i}.h5m"
        temp.export_dagmc_h5m_file(
            filename=str(part_h5m),
            min_mesh_size=0.5,
            max_mesh_size=2.0,
        )
        parts_h5m_files.append(part_h5m)
        print(f"Part {i}: OK -> {part_h5m.name}")
    except Exception as e:
        failed_parts_h5m.append((i, str(e)))
        print(f"Part {i}: FAIL - {e}")
        continue

print("\nSummary")
print(f"Successful parts: {len(parts_h5m_files)}")
print(f"Failed parts: {len(failed_parts_h5m)}")

for idx, err in failed_parts_h5m:
    print(f"  Part {idx}: {err}")

In [ ]:
print("\nSummary")
print(f"Successful parts: {len(parts_h5m_files)}")
print(f"Failed parts: {len(failed_parts_h5m)}")

for idx, err in failed_parts_h5m:
    print(f"  Part {idx}: {err}")

Check watertight and overlap of each parts

In [ ]:
# Run DAGMC watertight and overlap checks on each part file
for fh in parts_h5m_files:
    print(f"\n=== Checking {fh.name} ===")
    cmd_w = [str(dagmc_tools_path / "check_watertight"), str(fh)]
    subprocess.run(cmd_w)
    cmd_o = [str(dagmc_tools_path / "overlap_check"), str(fh)]
    subprocess.run(cmd_o)

Convert the h5m to vtk for visual inspection only!

In [ ]:
for h5m_file in parts_h5m_files:
    vtk_file = base_folder / f"{h5m_file.stem}.vtk"

    cmd = ["mbconvert", str(h5m_file), str(vtk_file)]
    subprocess.run(cmd, check=True)

    print(f"Converted {h5m_file.name} -> {vtk_file}")

Merge the single part.h5m into one h5m 

In [ ]:
# Combine all part .h5m files into a single .h5m using MOAB `mbconvert` 

merged_h5m = h5m_folder / f"{stp_file.stem}_merged.h5m"

print("\n===== COMBINE H5M FILES  =====")

mbconvert_path = shutil.which("mbconvert")

cmd = [mbconvert_path] + [str(f) for f in parts_h5m_files] + [str(merged_h5m)]

print(" ".join(cmd))
subprocess.run(cmd, check=True)

print(f"\n COMBINED H5M saved to: {merged_h5m}")

cmd = [
    str(dagmc_tools_path / "check_watertight"),
    str(merged_h5m)
]

subprocess.run(cmd)

cmd = [
    str(dagmc_tools_path / "overlap_check"),
    str(merged_h5m)
]

subprocess.run(cmd)



Export step as .vtk by part

In [ ]:
# Create per-part DAGMC .vtk files

parts_vtk_files = []
failed_parts_vtk = []

for i, part in enumerate(my_CadToDagmc.parts):
    try:
        temp = CadToDagmc()
        temp.parts = [part]

        part_vtk = vtk_folder / f"{stp_file.stem}_part_{i}.vtk"

        temp.export_unstructured_mesh_file(
            filename=str(part_vtk),

            min_mesh_size=5.0,
            max_mesh_size=20.0

            # min_mesh_size=0.5,
            # max_mesh_size=2.0

            # min_mesh_size=1.0,
            # max_mesh_size=4.0
        )

        parts_vtk_files.append(part_vtk)
        print(f"Part {i}: OK -> {part_vtk.name}")

    except Exception as e:
        failed_parts_vtk.append((i, str(e)))
        print(f"Part {i}: FAIL - {e}")
        continue

print("\nSummary")
print(f"Successful parts: {len(parts_vtk_files)}")
print(f"Failed parts: {len(failed_parts_vtk)}")

for idx, err in failed_parts_vtk:
    print(f"  Part {idx}: {err}")

In [ ]:
print("\nSummary")
print(f"Successful parts: {len(parts_vtk_files)}")
print(f"Failed parts: {len(failed_parts_vtk)}")

for idx, err in failed_parts_vtk:
    print(f"  Part {idx}: {err}")

Export again failed parts using a finer mesh

In [ ]:
parts_to_retry = [] # Change manually based on the summary failed parts

retry_failed_parts_vtk = []

for i in parts_to_retry:
    retry_part = my_CadToDagmc.parts[i]

    try:
        temp = CadToDagmc()
        temp.parts = [retry_part]

        part_vtk = vtk_folder / f"{stp_file.stem}_part_{i}.vtk"

        temp.export_unstructured_mesh_file(
            filename=str(part_vtk),
            min_mesh_size=0.5,
            max_mesh_size=2.0
        )

        parts_vtk_files.append(part_vtk)
        print(f"Part {i}: OK -> {part_vtk.name}")

    except Exception as e:
        retry_failed_parts_vtk.append((i, str(e)))
        print(f"Part {i}: FAIL - {e}")

print("\nSummary")
print(f"Successful parts: {len(parts_to_retry) - len(retry_failed_parts_vtk)}")
print(f"Failed parts: {len(retry_failed_parts_vtk)}")

for idx, err in retry_failed_parts_vtk:
    print(f"  Part {idx}: {err}")

Merge the part.vtk files in one vtk file

In [ ]:
# Combine all part .vtk files into a single .vtk using MOAB `mbconvert`

print("\n===== COMBINE VTK FILES =====")

merged_vtk = vtk_folder / f"{stp_file.stem}_merged.vtk"

mbconvert_path = shutil.which("mbconvert")

cmd = [mbconvert_path] + [str(f) for f in parts_vtk_files] + [str(merged_vtk)]

print(" ".join(cmd))
subprocess.run(cmd, check=True)

print(f"\n COMBINED VTK saved to: {merged_vtk}")